In [18]:
"""
Clinical Trial Eligibility Criteria Processing Pipeline
========================================================
A three-stage pipeline for processing clinical trial eligibility criteria:
1. Pre-classification: Decompose complex criteria into atomic sub-criteria
2. Classification: Extract structured attributes from sub-criteria
3. Question Generation: Transform criteria into patient-friendly questionnaires

Each stage can be run independently or as part of the complete pipeline.
"""

import os
import logging
import pandas as pd
import time
import ast
import re
import json
import hashlib
from typing import List, Tuple, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI


# ================================================================
# SETUP & CONFIGURATION
# ================================================================

# Environment setup
os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

# Logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# Directory structure
ROOT_DIR = "../"
INPUT_DIR = "data/"
OUTPUT_DIR_PRECLSF = "output/preclsf/"
OUTPUT_DIR_CLSF = "output/clsf/"
OUTPUT_DIR_QUES = "output/question/"
CHECKPOINT_SUBDIR = "checkpoints"

# model configuration
model_MAP = {
    'gpt-5.1': 'gpt-5.1',
    'gpt-5-mini': 'gpt-5-mini',
    'gpt-5-nano': 'gpt-5-nano',
    'gpto3': 'o3-mini',
    'gpt4oft': 'ft:gpt-4o-mini-2024-07-18:personal:preclsf-1216:AewDsdym'
}

LLM_PARAMS = {
    'max_tokens': 2000,
    'num_retries': 2,
    'delay_time': 10,
}

# Disease mappings
DISEASE_DICT = {
    'Alzheimer': 'Alzheimer disease',
    'Diabetes': 'diabetes',
    'NSCLC': 'non-small-cell lung cancer',
    'UC': 'ulcerative colitis',
    'AML': 'acute myeloid leukemia',
    'CF': 'cystic fibrosis',
    'MS': 'multiple sclerosis',
}

TYPE_SWAP = {
    'inclusion': 'exclusion',
    'exclusion': 'inclusion'
}


# ================================================================
# SHARED HELPER FUNCTIONS
# ================================================================

def ensure_dir(path: str) -> None:
    """Create directory if it doesn't exist."""
    os.makedirs(path, exist_ok=True)


def short_hash(text: str, n: int = 8) -> str:
    """Generate short hash for identification."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n]


def safe_filename(name: str) -> str:
    """Convert string to safe filename."""
    name = str(name)
    name = re.sub(r"\s+", "_", name.strip())
    name = re.sub(r"[^a-zA-Z0-9_\-\.]", "", name)
    return name[:120] if len(name) > 120 else name

def strip_code_fences(text: str) -> str:
    if text.startswith("```"):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    return text.strip()

def call_openai_with_retry(
    prompt: str,
    model: str,
    system_message: str,
    max_completion_tokens: int = 6000,
    retries: int = 2
) -> str:
    """Call OpenAI API with retry logic."""
    prompt_id = short_hash(prompt)
    last_exception = None
    delay = LLM_PARAMS.get('delay_time', 10)

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": prompt}
                ],
                max_completion_tokens=max_completion_tokens
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            last_exception = e
            logging.warning(f"Attempt {attempt+1}/{retries} failed | prompt_id={prompt_id} | error={e}")
            if attempt < retries - 1:
                time.sleep(delay)
    
    logging.error(f"All retries failed | prompt_id={prompt_id}")
    raise last_exception


# ================================================================
# STAGE 1: PRE-CLASSIFICATION
# ================================================================

def build_preclsf_prompt(criterion: str, criterion_type: str, disease: str) -> str:
    """Build prompt for pre-classification stage."""
    instructions = f"""
# Objective
Break down complex clinical trial eligibility criteria into atomic sub-criteria and classify their logical relationships and categories. Everything you do is to extract structured information from clinical trial eligibility criteria.
"""
    
    steps = f"""
# Instructions
## Step 1: Identify Sub-criteria
- Decompose criteria containing multiple conditions, nested logic, or complex sentences
- Each sub-criterion must be self-explanatory with all essential information
- Remove supplemental information (e.g., guideline references, PI evaluation notes, other non-essential bracketed clarifications)
- Remove filler phrases but keep medical qualifiers
- Do NOT separate if:
-- Sentences define or clarify the previous statement
-- Clauses are connected by commas without adding complexity
-- Related concepts are better put together (e.g., don't separate "history or presence of infection" into two sub-criteria)
-- Sentences are interdependent and separation would lose meaning
-- The criterion is already atomic (simple, single condition)

## Step 2: Determine Logical Relationships
Classify the relationship between sub-criteria:
- 'All': All sub-criteria must be met (AND logic)
- 'Any': Any one sub-criterion is sufficient (OR logic)
- 'Example': Sub-criterion illustrates the main criterion (e.g., "such as", "including")
- 'Condition': Sub-criterion defines when the main criterion applies (e.g., "if", "for patients with")
- 'Exception': Sub-criterion negates the main criterion (e.g., "except", "unless")
- 'Standalone': Criterion has no sub-criteria (use 'All' by default)
- Edge Case
-- For Inclusion Criterion: "Female and male" = ANY gender qualifies (OR logic)
-- For Exclusion Criterion: "COVID-19 and other infectious diseases" = ANY condition excludes (OR logic)

## Step 3: Assign each sub-criterion to ONE primary category and secondary sub-category:
- Demographics: Age, Gender, Ethnicity
- Diagnosis: Diagnosis, Severity, Biomarkers
- History: Cardiovascular, Kidney, Neurological, Endocrine, Gastrointestinal, Infectious_Disease, Malignancy, Autoimmune, Other_Complication
- Measures: Laboratory, Vital_Signs, Body_Measures
- Treatment: Current_Treatment, Treatment_History, Medication_Exclusion, Immunosuppression, Device_Use, Transplant, Vaccination, Adverse_Events
- Contraception: Pregnancy, Contraception, Lactation
- Lifestyle: Physical_Activity, Substance_Use, Nutrition, Cognitive_Function
- Participation: Language, Geographic, Logistics, General_Health, Others

# Output Format
[
    ['Primary_Category_1', 'Secondary_Category_1', 'Sub-Criterion_1', 'Logic_1'],
    ['Primary_Category_2', 'Secondary_Category_2', 'Sub-Criterion_2', 'Logic_2'],
    ...
]

If criterion requires no decomposition:
[
    ['Primary_Category', 'Secondary_Category', '', 'Standalone']
]

# Examples
Example 1: No decomposition needed 
Input: "Type 2 Diabetes"
[
    ['Diagnosis', 'Diagnosis', '', 'Standalone']
]

Example 2: Simple AND logic 
Input: "Male or female persons ≥40 years old with T1D"
[
    ['Demographics', 'Age', '≥40 years old', 'All'],
    ['Diagnosis', 'Diagnosis', 'T1D', 'All']
]

Example 3: Complex nested logic 
Input: "Presence of chronic kidney disease (UACR >30 mg/g or eGFR < 60 ml/min/1.73 m2) OR history of ischemic heart disease (previous myocardial infarction, stroke or angina)"
[
    ['History', 'Kidney', 'Chronic kidney disease with UACR >30 mg/g', 'Any'],
    ['History', 'Kidney', 'Chronic kidney disease with eGFR < 60 ml/min/1.73 m2', 'Any'],
    ['History', 'Cardiovascular', 'History of ischemic heart disease with previous myocardial infarction', 'Any'],
    ['History', 'Cardiovascular', 'History of ischemic heart disease with previous stroke', 'Any'],
    ['History', 'Cardiovascular', 'History of ischemic heart disease with angina', 'Any']
]
"""
    
    return instructions + "\nCriterion:\n" + criterion + "\n" + steps


def preclsf_stage(disease: str, model: str, test_mode: bool = False) -> str:
    """
    Stage 1: Pre-classification
    Breaks down complex criteria into atomic sub-criteria.
    Returns path to output file.
    """
    logging.info(f"[STAGE 1: PRE-CLASSIFICATION] Processing {disease}")
    
    input_file = os.path.join(ROOT_DIR, INPUT_DIR, f"{disease}.csv")
    output_file = os.path.join(ROOT_DIR, OUTPUT_DIR_PRECLSF, f"preclsf_{model}_{disease}.xlsx")
    ensure_dir(os.path.join(ROOT_DIR, OUTPUT_DIR_PRECLSF))
    
    # Check if input file exists
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")
    
    # Load input
    df_criteria = pd.read_csv(input_file)
    df_criteria['Criterion ID'] = range(1, len(df_criteria) + 1)
    
    # Prepare output dataframe
    df_subcriteria = pd.DataFrame(columns=[
        'Criterion ID', 'Primary Category', 'Secondary Category',
        'Sub-Criterion', 'Type', 'Outer Logic', 'Criterion'
    ])
    
    # Process criteria
    subset = df_criteria.head(13) if test_mode else df_criteria
    for idx, row in subset.iterrows():
        logging.info(f"Processing criterion {idx+1}/{len(subset)}: nct_id={row['nct_id']}")
        
        prompt = build_preclsf_prompt(row['Criterion'], row['Type'], disease)
        output_text = call_openai_with_retry(
            prompt, model,
            "You are a medical criteria extraction expert. Return only valid Python list format.",
            retries=LLM_PARAMS['num_retries']
        )
        
        try:
            response = eval(output_text)
            columns = ['Primary Category', 'Secondary Category', 'Sub-Criterion', 'Outer Logic']
            df_response = pd.DataFrame(response, columns=columns)
            df_response['Criterion ID'] = row['Criterion ID']
            df_response['Type'] = row['Type']
            df_response['Criterion'] = row['Criterion']
            df_subcriteria = pd.concat([df_subcriteria, df_response], ignore_index=True)
        except Exception as e:
            logging.error(f"Failed to parse response for criterion {row['Criterion ID']}: {e}")
            fallback = pd.DataFrame([[
                row['Criterion ID'], '', '', '', row['Type'], 'Standalone', row['Criterion']
            ]], columns=df_subcriteria.columns)
            df_subcriteria = pd.concat([df_subcriteria, fallback], ignore_index=True)
        
        time.sleep(1)
    
    # Post-processing
    decomposed_criteria = df_subcriteria[
        (df_subcriteria['Outer Logic'] != 'Standalone') | 
        (df_subcriteria['Sub-Criterion'] != '')
    ]['Criterion ID'].unique()
    
    df_subcriteria['multi'] = df_subcriteria['Criterion ID'].apply(
        lambda x: 1 if x in decomposed_criteria else 0
    )
    
    df_subcriteria['Sub-Criterion'] = df_subcriteria.apply(
        lambda x: x['Criterion'] if x['Sub-Criterion'] == "" else x['Sub-Criterion'], 
        axis=1
    )
    
    df_subcriteria.insert(0, 'Sub-Criterion ID', range(1, len(df_subcriteria) + 1))
    
    # Update original criteria dataframe
    multi_map = df_subcriteria.drop_duplicates('Criterion ID').set_index('Criterion ID')['multi'].to_dict()
    df_criteria['multi'] = df_criteria['Criterion ID'].map(multi_map).fillna(0).astype(int)
    df_criteria['Extraction Wrong'] = ""
    
    # Save output
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_subcriteria.to_excel(writer, sheet_name='subcriterion', index=False)
        df_criteria.to_excel(writer, sheet_name='criterion', index=False)
    
    logging.info(f"[STAGE 1 COMPLETE] Output: {output_file}")
    return output_file


# ================================================================
# STAGE 2: CLASSIFICATION
# ================================================================

def build_clsf_prompt(criterion: str, criterion_type: str, disease: str) -> str:
    """Build prompt for classification stage."""
    instructions = f"""
# Objective
Extract structured, atomic attributes from the sub-criterion to enable precise eligibility questionnaire generation.

Disease context: {DISEASE_DICT.get(disease, disease)}
"""
    
    steps = """
# Instructions
## Step 1: Extract Core Attribute
- Identify the primary medical concept or measurable entity
- Remove non-restrictive clauses, definitions, and guideline references
- Keep restrictive clauses essential to meaning
- Make attributes self-sufficient (e.g., "diagnosis of diabetes" not just "diagnosis")

## Step 2: Extract Modifiers
Capture ALL qualifiers that affect eligibility determination:
- Status: Active, inactive, stable, controlled, severe, chronic, acute
- Severity: Mild, moderate, severe, stage I-V, NYHA class
- Specificity: Subtypes, variants, specific conditions
- Measurement: Numerical values, ranges, thresholds, units
- Binary: Yes/No, positive/negative, present/absent
- Temporal: Ongoing, previous, planned, history of
- Format: Comma-separated list. Include ALL relevant modifiers even if lengthy.

## Step 3: Extract Time Interval
Capture any temporal restrictions:
- Absolute times: "within 30 days", "at least 1 year", "more than 6 months"
- Relative times: "before screening", "prior to enrollment", "after last dose"
- Duration: "for at least 2 months", "during study period"
- Timing: "at the time of screening", "before randomization"
- Format: Single string preserving original temporal relationship

## Step 4: Extract Condition
Identify if the criterion applies conditionally:
- Population: "For women", "In patients with diabetes", "Japanese participants"
- Context: "If diagnosed after age 30", "When using insulin pump"
- Situational: "During pregnancy", "At baseline"

## Step 5: Decide Logic
- 'All': All sub-criteria must be met (AND logic)
- 'Any': Any one sub-criterion is sufficient (OR logic)
- 'Example': Sub-criterion illustrates the main criterion
- 'Condition': Sub-criterion defines when the main criterion applies
- 'Exception': Sub-criterion negates the main criterion
- 'Standalone': Criterion has no sub-criteria

# Output Format
[
    ['Attribute_1', 'Modifier_1', 'Time_Interval_1', 'Condition_1', 'Logic_1'],
    ['Attribute_2', 'Modifier_2', 'Time_Interval_2', 'Condition_2', 'Logic_2'],
    ...
]

Use "" (empty string) for fields that don't apply.
Return ONLY the list, no explanatory text.

# Examples
Example 1: Simple attribute with threshold
Input: "HbA1c <= 9.0%"
[
    ['HbA1c', '≤9.0%', '', '', 'All']
]

Example 2: Attribute with time interval
Input: "Treatment with systemic corticosteroids within 8 weeks before screening"
[
    ['Treatment with systemic corticosteroids', '', 'within 8 weeks before screening', '', 'All']
]

Example 3: Complex attribute with multiple modifiers
Input: "Recurrent severe hypoglycemic episodes requiring assistance within the last year"
[
    ['Hypoglycemic episodes', 'recurrent, severe, requiring assistance', 'within the last year', '', 'All']
]
"""
    
    return instructions + steps + f"\n\nExtract attributes from this {criterion_type} criterion:\n{criterion}"


def safe_parse_response(response_text: str) -> list:
    """Safely parse API response to list of lists."""
    response_text = response_text.strip()
    response_text = response_text.replace('"', '"').replace('"', '"').replace("'", "'").replace("'", "'")
    
    if response_text.startswith('```'):
        lines = response_text.split('\n')
        response_text = '\n'.join(lines[1:-1]) if len(lines) > 2 else response_text
    
    response_text = response_text.strip()
    
    try:
        parsed = ast.literal_eval(response_text)
        
        if not isinstance(parsed, list):
            logging.warning(f"Response is not a list: {type(parsed)}")
            return [["", "", "", "", "Standalone"]]
        
        parsed_fixed = []
        for i, lst in enumerate(parsed):
            if not isinstance(lst, list):
                continue
            
            lst = [str(item) if item is not None else "" for item in lst]
            
            if len(lst) < 5:
                lst = lst + [''] * (5 - len(lst))
            elif len(lst) > 5:
                lst = lst[:5]
            
            parsed_fixed.append(lst)
        
        return parsed_fixed if parsed_fixed else [["", "", "", "", "Standalone"]]
    
    except (ValueError, SyntaxError) as e:
        logging.error(f"Failed to parse API response: {e}")
        return [["", "", "", "", "Standalone"]]


def clsf_stage(disease: str, model: str, test_mode: bool = False) -> str:
    """
    Stage 2: Classification
    Extracts structured attributes from sub-criteria.
    Returns path to output file.
    """
    logging.info(f"[STAGE 2: CLASSIFICATION] Processing {disease}")
    
    input_file = os.path.join(ROOT_DIR, OUTPUT_DIR_PRECLSF, f"preclsf_{model}_{disease}.xlsx")
    output_file = os.path.join(ROOT_DIR, OUTPUT_DIR_CLSF, f"clsf_{model}_{disease}.xlsx")
    ensure_dir(os.path.join(ROOT_DIR, OUTPUT_DIR_CLSF))
    
    # Check if input file exists
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")
    
    # Load input
    df_subcriteria = pd.read_excel(input_file, sheet_name='subcriterion')
    logging.info(f"Loaded {len(df_subcriteria)} sub-criteria")
    
    # Prepare output dataframe
    df_attributes = pd.DataFrame(columns=[
        'Sub-Criterion ID', 'Criterion ID', 'Primary Category', 'Secondary Category',
        'Sub-Criterion', 'Attribute', 'Modifier', 'Time_Interval', 'Condition',
        'Type', 'Outer Logic', 'Logic', 'Criterion', 'multi'
    ])
    
    # Process sub-criteria
    subset = df_subcriteria.head(5) if test_mode else df_subcriteria
    for idx, row in subset.iterrows():
        logging.info(f"Processing sub-criterion {idx+1}/{len(subset)} (ID: {row['Sub-Criterion ID']})")
        
        prompt = build_clsf_prompt(row['Sub-Criterion'], row['Type'], disease)
        output_text = call_openai_with_retry(
            prompt, model,
            "You are a medical criteria extraction expert. Return only the requested Python list format with no additional text.",
            retries=LLM_PARAMS['num_retries']
        )
        
        attributes = safe_parse_response(output_text)
        
        df_attrs = pd.DataFrame(
            attributes,
            columns=['Attribute', 'Modifier', 'Time_Interval', 'Condition', 'Logic']
        )
        
        # Add metadata
        for col in ['Sub-Criterion ID', 'Criterion ID', 'Primary Category', 'Secondary Category',
                    'Sub-Criterion', 'Type', 'Outer Logic', 'Criterion', 'multi']:
            df_attrs[col] = row[col]
        
        # Reorder columns
        df_attrs = df_attrs[[
            'Sub-Criterion ID', 'Criterion ID', 'Primary Category', 'Secondary Category',
            'Sub-Criterion', 'Attribute', 'Modifier', 'Time_Interval', 'Condition',
            'Type', 'Outer Logic', 'Logic', 'Criterion', 'multi'
        ]]
        
        df_attributes = pd.concat([df_attributes, df_attrs], ignore_index=True)
        time.sleep(1)
    
    # Post-processing
    df_attributes.insert(0, 'Attribute ID', range(1, len(df_attributes) + 1))
    
    decomposed_subcriteria = df_attributes.groupby('Sub-Criterion ID').size()
    decomposed_subcriteria = decomposed_subcriteria[decomposed_subcriteria > 1].index.tolist()
    
    df_attributes['attribute_multi'] = df_attributes['Sub-Criterion ID'].apply(
        lambda x: 1 if x in decomposed_subcriteria else 0
    )
    
    # Save output
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_attributes.to_excel(writer, sheet_name='attributes', index=False)
        df_subcriteria.to_excel(writer, sheet_name='subcriteria', index=False)
    
    logging.info(f"[STAGE 2 COMPLETE] Output: {output_file}")
    logging.info(f"Extracted {len(df_attributes)} attributes from {len(df_subcriteria)} sub-criteria")
    
    return output_file


# ================================================================
# STAGE 3: QUESTION GENERATION
# ================================================================

def build_ques_prompt(criterion: str, criterion_type: str, disease: str) -> str:
    """Build prompt for question generation stage."""
    instructions = """
# Objective
Transform atomic eligibility criteria into a consolidated questionnaire that:
- Minimizes respondent burden by eliminating redundancy
- Maintains precision for accurate eligibility determination
- Provides intuitive, user-friendly questions
- Ensures complete criterion coverage with proper logic mapping
"""
    
    steps = """
# Instructions
## Step 1: Review all criteria and identify underlying medical concepts

## Step 2: Determine Question Format
- Radio Buttons (mutually exclusive): Age ranges, diabetes type, pregnancy status
- Checkboxes (multiple selection): Comorbidities, medications, symptoms

## Step 3: Design Comprehensive Option Sets
- For Numeric Ranges: Create mutually exclusive ranges covering all thresholds
- For Categorical Data: Order by specificity
- For Temporal Data: Create time buckets covering all thresholds
- For Hierarchical Conditions: List specific conditions first
- For Binary Criteria: Expand Yes/No into informative options

## Step 4: Map Criteria to Options
For each option, create dictionary mapping:
- Key: Criterion ID number
- Value: "Y" = meets criterion, "N" = violates criterion

## Step 5: Special Cases
- Numeric criteria: Pay attention to ≥, ≤, ranges
- "None of the Above": Must map ALL relevant criteria to "N"

# Output Format
[
    ["Question text?", [
        ["Option A description", {"CriterionID1": "Y", "CriterionID2": "N"}],
        ["Option B description", {"CriterionID1": "N", "CriterionID3": "Y"}],
        ["None of the above", {"CriterionID1": "N", "CriterionID2": "N", "CriterionID3": "N"}]
    ]],
    ["Next question?", [...]]
]

# Examples
Example 1: Numeric range
Input: (15) HbA1c ≤9.0%, (23) HbA1c 7.0-10.0%
[
    ["What is your most recent HbA1c level?", [
        ["Less than 7.0%", {"15": "Y", "23": "N"}],
        ["7.0-9.0%", {"15": "Y", "23": "Y"}],
        ["9.0-10.0%", {"15": "N", "23": "Y"}],
        ["Greater than 10.0%", {"15": "N", "23": "N"}],
        ["Don't know", {}]
    ]]
]

Example 2: Hierarchical conditions
Input: (5) Cardiovascular disease, (12) History of MI, (18) History of stroke
[
    ["Do you have any cardiovascular conditions?", [
        ["Previous heart attack (myocardial infarction)", {"5": "Y", "12": "Y"}],
        ["Previous stroke", {"5": "Y", "18": "Y"}],
        ["Other cardiovascular disease", {"5": "Y"}],
        ["None of the above", {"5": "N", "12": "N", "18": "N"}]
    ]]
]
"""
    
    return instructions + steps + "\n\n" + criterion


def snippet_from_questions(questions: List[Any], max_len: int = 160) -> str:
    """Generate preview snippet from questions."""
    if not questions or not isinstance(questions, list):
        return "(no questions)"
    
    first = questions[0]
    if not isinstance(first, list) or len(first) < 2:
        return "(unexpected format)"
    
    q_text = str(first[0])
    options = first[1] if isinstance(first[1], list) else []
    opt0 = str(options[0][0]) if options and isinstance(options[0], list) and options[0] else ""
    
    q_text_short = (q_text[:max_len] + "…") if len(q_text) > max_len else q_text
    opt0_short = (opt0[:max_len] + "…") if len(opt0) > max_len else opt0
    
    return f"Q: {q_text_short} | #opts={len(options)} | opt0: {opt0_short}"


def write_category_checkpoint(
    checkpoint_dir: str,
    disease: str,
    model: str,
    idx: int,
    category: str,
    questions: List[Any],
    all_questions: List[Any],
    df_attribute: pd.DataFrame,
    df_subcriterion: pd.DataFrame
) -> None:
    """Save per-category checkpoint with progress info."""
    ensure_dir(checkpoint_dir)
    
    cat_safe = safe_filename(category)
    per_cat_json = os.path.join(
        checkpoint_dir,
        f"{model}_{disease}__{idx:03d}__{cat_safe}.json"
    )
    
    with open(per_cat_json, "w", encoding="utf-8") as f:
        json.dump(questions, f, ensure_ascii=False, indent=2)
    
    # Rolling checkpoint
    rolling_xlsx = os.path.join(checkpoint_dir, f"{model}_{disease}__rolling_questions.xlsx")
    df_question_partial = pd.DataFrame(all_questions, columns=['Question', 'Options', 'Category'])
    df_question_partial.insert(0, 'Question ID', range(len(df_question_partial)))
    
    with pd.ExcelWriter(rolling_xlsx, engine='openpyxl') as writer:
        df_question_partial.to_excel(writer, sheet_name='question', index=False)
        df_attribute.to_excel(writer, sheet_name='attribute', index=False)
        df_subcriterion.to_excel(writer, sheet_name='subcriterion', index=False)
    
    print(
        f"[CHECKPOINT] {disease} | {model} | {idx:03d} '{category}' "
        f"| +{len(questions)} questions | {snippet_from_questions(questions)}"
    )
    
    logging.info(f"Checkpoint saved | rolling={rolling_xlsx} | total={len(df_question_partial)}")


def simplify_option_text(option_text: str, model: str) -> str:
    """Simplify medical language for patient-friendly options."""
    if option_text.lower() in ['none of the above', 'none of the above.', "don't know", 'unknown']:
        return option_text
    
    prompt = f"Simplify the following medical questionnaire option:\n\nOriginal: {option_text}\n\nReturn ONLY the simplified text."
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You simplify medical language for patients."},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.choices[0].message.content.strip().strip('"\'').strip()


def ques_stage(disease: str, model: str) -> str:
    """
    Stage 3: Question Generation
    Transforms criteria into patient questionnaires.
    Returns path to output file.
    """
    logging.info(f"[STAGE 3: QUESTION GENERATION] Processing {disease}")
    
    input_file = os.path.join(ROOT_DIR, OUTPUT_DIR_CLSF, f"clsf_{model}_{disease}.xlsx")
    output_file = os.path.join(ROOT_DIR, OUTPUT_DIR_QUES, f"ques_{model}_{disease}.xlsx")
    ensure_dir(os.path.join(ROOT_DIR, OUTPUT_DIR_QUES))
    
    # Check if input file exists
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")
    
    checkpoint_dir = os.path.join(ROOT_DIR, OUTPUT_DIR_QUES, CHECKPOINT_SUBDIR, f"{model}_{disease}")
    ensure_dir(checkpoint_dir)
    
    # Load input
    df_subcriterion = pd.read_excel(input_file, sheet_name='subcriteria')
    df_attribute = pd.read_excel(input_file, sheet_name='attributes')
    
    df_attribute['Attribute'] = df_attribute['Attribute'].fillna("")
    df_attribute['Modifier'] = df_attribute['Modifier'].fillna("")
    
    df_attribute['Full Attribute'] = (
        df_attribute['Attribute ID'].astype(str) + " | " +
        df_attribute['Attribute'] + " | " +
        df_attribute['Modifier']
    )
    
    categories = df_attribute['Secondary Category'].dropna().astype(str).unique().tolist()
    logging.info(f"Found {len(categories)} categories")
    
    all_questions: List[Any] = []
    
    # Process each category
    for idx, category in enumerate(categories, start=1):
        logging.info(f"{idx}/{len(categories)}: {category}")
        
        df_cat_full = df_attribute[df_attribute['Secondary Category'].astype(str) == str(category)]
        
        if df_cat_full.empty:
            raise ValueError(f"No attributes for category: {category}")
        
        outer_type = str(df_cat_full['Outer Logic'].iloc[0]).strip()
        df_category = df_cat_full[['Full Attribute']]
        
        prompt = build_ques_prompt(
            criterion=df_category.to_string(index=False),
            criterion_type=outer_type,
            disease=disease
        )
        

        for attempt in range(4):   # 1 initial + 2 retries
            try:
                output_text = call_openai_with_retry(
                    prompt,
                    model,
                    "You are a medical expert helping to create clinical trial eligibility questionnaires. Respond with VALID JSON ONLY.",
                    retries=LLM_PARAMS["num_retries"]
                )

                output_text = strip_code_fences(output_text)
                questions = json.loads(output_text)   # <- this is what may fail
                break                                 # success, exit loop

            except json.JSONDecodeError as e:
                last_error = e
                print(f"\n===== JSON parse failed (attempt {attempt+1}/3) =====")
                print(output_text)
                print("==============================================\n")

        # After loop
        if questions is None:
            raise RuntimeError("LLM failed to return valid JSON after 3 attempts") from last_error

        if not isinstance(questions, list):
            raise ValueError(f"Expected list JSON but got {type(questions)}")
        
        # Attach category
        for q in questions:
            q.append(category)
        
        all_questions.extend(questions)
        
        # Checkpoint
        write_category_checkpoint(
            checkpoint_dir, disease, model, idx, category,
            questions, all_questions, df_attribute, df_subcriterion
        )
        
        time.sleep(1)
    
    # Create question dataframe
    df_question = pd.DataFrame(all_questions, columns=['Question', 'Options', 'Category'])
    df_question.insert(0, 'Question ID', range(len(df_question)))
    
    # Save questions-only file early
    question_only_path = output_file.replace(".xlsx", "_questions_only.xlsx")
    with pd.ExcelWriter(question_only_path, engine='openpyxl') as writer:
        df_question.to_excel(writer, sheet_name='question', index=False)
        df_attribute.to_excel(writer, sheet_name='attribute', index=False)
        df_subcriterion.to_excel(writer, sheet_name='subcriterion', index=False)
    
    logging.info(f"Saved questions | rows={len(df_question)} | path={question_only_path}")
    
    # Generate option mappings
    logging.info("Generating option mappings...")
    
    df_option = df_question.explode('Options').reset_index(drop=True)
    df_option['Option Text'] = df_option['Options'].apply(lambda x: x[0])
    df_option['Attribute Mapping'] = df_option['Options'].apply(lambda x: x[1])
    
    # Simplify option text
    logging.info(f"Simplifying {len(df_option)} options...")
    df_option['Curated Option'] = [
        simplify_option_text(text, model)
        for text in df_option['Option Text']
    ]
    
    # Expand attribute mappings
    expanded_rows = []
    for _, row in df_option.iterrows():
        mapping = row['Attribute Mapping']
        if isinstance(mapping, dict) and mapping:
            for attr_id, value in mapping.items():
                r = row.copy()
                r['Attribute ID'] = str(attr_id)
                r['Attribute Value'] = value
                expanded_rows.append(r)
        else:
            r = row.copy()
            r['Attribute ID'] = None
            r['Attribute Value'] = None
            expanded_rows.append(r)
    
    df_option_attr = pd.DataFrame(expanded_rows)
    
    # Merge with attributes
    df_attribute['Attribute ID'] = df_attribute['Attribute ID'].astype('string')
    df_option_attr['Attribute ID'] = df_option_attr['Attribute ID'].astype('string')
    
    df_option_attr = df_option_attr.merge(
        df_attribute[[
            'Attribute ID', 'Criterion ID', 'Criterion',
            'Logic', 'Outer Logic', 'Full Attribute'
        ]],
        on='Attribute ID',
        how='left'
    )
    
    # Save final output
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_question.to_excel(writer, sheet_name='question', index=False)
        df_option.to_excel(writer, sheet_name='option', index=False)
        df_option_attr.to_excel(writer, sheet_name='option_attribute', index=False)
        df_attribute.to_excel(writer, sheet_name='attribute', index=False)
        df_subcriterion.to_excel(writer, sheet_name='subcriterion', index=False)
    
    logging.info(f"[STAGE 3 COMPLETE] Output: {output_file}")
    return output_file


# ================================================================
# MAIN EXECUTION
# ================================================================

def main():
    """
    Run complete pipeline or specific stages for specified diseases.
    
    Configure which stages to run by setting the flags below:
    - RUN_PRECLSF: Run pre-classification stage
    - RUN_CLSF: Run classification stage
    - RUN_QUES: Run question generation stage
    
    Set all to True to run the complete pipeline.
    Set specific stages to True to run only those stages.
    """
    
    # ============================================================
    # CONFIGURATION - EDIT THESE VALUES
    # ============================================================
    
    # Disease configuration
    DISEASES = ['AML']  # Add more: 'Diabetes', 'NSCLC', 'UC', 'CF', 'MS', 'Alzheimer'
    MODELS = ['gpt-5-mini']# ,  'gpt-5.1'
    TEST_MODE = False  # Set to True to process only first few records
    
    # Stage selection - Set to True to run that stage
    RUN_PRECLSF = False   # Stage 1: Pre-classification
    RUN_CLSF = False       # Stage 2: Classification
    RUN_QUES = True       # Stage 3: Question Generation
    
    # ============================================================
    # END CONFIGURATION
    # ============================================================
    
    # Validate configuration
    if not any([RUN_PRECLSF, RUN_CLSF, RUN_QUES]):
        logging.error("No stages selected to run! Set at least one of RUN_PRECLSF, RUN_CLSF, RUN_QUES to True.")
        return
    
    # Display configuration
    stages_to_run = []
    if RUN_PRECLSF:
        stages_to_run.append("Pre-classification")
    if RUN_CLSF:
        stages_to_run.append("Classification")
    if RUN_QUES:
        stages_to_run.append("Question Generation")
    
    for model in MODELS:
        logging.info(f"\n{'='*80}")
        logging.info(f"PIPELINE CONFIGURATION")
        logging.info(f"{'='*80}")
        logging.info(f"Diseases: {', '.join(DISEASES)}")
        logging.info(f"MODEL: {model}")
        logging.info(f"Test Mode: {TEST_MODE}")
        logging.info(f"Stages to run: {' → '.join(stages_to_run)}")
        logging.info(f"{'='*80}\n")
        
        # Process each disease
        for disease in DISEASES:
            logging.info(f"\n{'='*80}")
            logging.info(f"PROCESSING DISEASE: {disease}")
            logging.info(f"{'='*80}\n")
            
            try:
                # Stage 1: Pre-classification
                if RUN_PRECLSF:
                    preclsf_output = preclsf_stage(disease, model, test_mode=TEST_MODE)
                    logging.info(f"Stage 1 output: {preclsf_output}\n")
                else:
                    logging.info("[STAGE 1: PRE-CLASSIFICATION] Skipped\n")
                
                # Stage 2: Classification
                if RUN_CLSF:
                    clsf_output = clsf_stage(disease, model, test_mode=TEST_MODE)
                    logging.info(f"Stage 2 output: {clsf_output}\n")
                else:
                    logging.info("[STAGE 2: CLASSIFICATION] Skipped\n")
                
                # Stage 3: Question Generation
                if RUN_QUES:
                    ques_output = ques_stage(disease, model)
                    logging.info(f"Stage 3 output: {ques_output}\n")
                else:
                    logging.info("[STAGE 3: QUESTION GENERATION] Skipped\n")
                
                logging.info(f"\n{'='*80}")
                logging.info(f"COMPLETED PROCESSING FOR {disease}")
                logging.info(f"{'='*80}\n")
                
            except FileNotFoundError as e:
                logging.error(f"File not found for {disease}: {e}")
                logging.error("Make sure previous stage outputs exist if running later stages.")
                continue
            except Exception as e:
                logging.error(f"Pipeline failed for {disease}: {e}", exc_info=True)
                continue
        
        logging.info("All processing complete.")


if __name__ == "__main__":
    main()

2025-12-25 17:53:31,388 [INFO] 
2025-12-25 17:53:31,389 [INFO] PIPELINE CONFIGURATION
2025-12-25 17:53:31,389 [INFO] ================================================================================
2025-12-25 17:53:31,389 [INFO] Diseases: AML
2025-12-25 17:53:31,390 [INFO] MODEL: gpt-5-mini
2025-12-25 17:53:31,390 [INFO] Test Mode: False
2025-12-25 17:53:31,391 [INFO] Stages to run: Question Generation
2025-12-25 17:53:31,391 [INFO] ================================================================================

2025-12-25 17:53:31,391 [INFO] 
2025-12-25 17:53:31,391 [INFO] PROCESSING DISEASE: AML
2025-12-25 17:53:31,392 [INFO] ================================================================================

2025-12-25 17:53:31,392 [INFO] [STAGE 1: PRE-CLASSIFICATION] Skipped

2025-12-25 17:53:31,392 [INFO] [STAGE 2: CLASSIFICATION] Skipped

2025-12-25 17:53:31,392 [INFO] [STAGE 3: QUESTION GENERATION] Processing AML
2025-12-25 17:53:31,530 [INFO] Found 30 categories
2025-12-25 17:53:

[CHECKPOINT] AML | gpt-5-mini | 001 'Diagnosis' | +8 questions | Q: Which of the following best describes your hematologic diagnosis? (Select the single best answer) | #opts=11 | opt0: Acute promyelocytic leukemia (APL / M3, e.g., PML-RARA–positive)


2025-12-25 17:55:30,696 [INFO] 2/30: Current_Treatment
2025-12-25 17:56:59,582 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 17:56:59,791 [INFO] Checkpoint saved | rolling=../output/question/checkpoints/gpt-5-mini_AML/gpt-5-mini_AML__rolling_questions.xlsx | total=15


[CHECKPOINT] AML | gpt-5-mini | 002 'Current_Treatment' | +7 questions | Q: Are you currently receiving any hypomethylating agent combined with venetoclax (VEN)? Select all that apply. | #opts=4 | opt0: Azacitidine plus venetoclax (currently receiving)


2025-12-25 17:57:00,797 [INFO] 3/30: Biomarkers
2025-12-25 17:58:26,353 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 17:58:26,560 [INFO] Checkpoint saved | rolling=../output/question/checkpoints/gpt-5-mini_AML/gpt-5-mini_AML__rolling_questions.xlsx | total=21


...

2025-12-25 18:25:59,430 [INFO] 29/30: Cognitive_Function
2025-12-25 18:26:27,216 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:26:27,403 [INFO] Checkpoint saved | rolling=../output/question/checkpoints/gpt-5-mini_AML/gpt-5-mini_AML__rolling_questions.xlsx | total=78


[CHECKPOINT] AML | gpt-5-mini | 029 'Cognitive_Function' | +1 questions | Q: Do you currently have any psychological or psychiatric condition that could interfere with study participation, compliance, or compromise safety (as judged by a… | #opts=8 | opt0: Active psychosis or schizophrenia with recent exacerbation


2025-12-25 18:26:28,414 [INFO] 30/30: Others
2025-12-25 18:26:45,234 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:26:45,396 [INFO] Checkpoint saved | rolling=../output/question/checkpoints/gpt-5-mini_AML/gpt-5-mini_AML__rolling_questions.xlsx | total=79


[CHECKPOINT] AML | gpt-5-mini | 030 'Others' | +1 questions | Q: Which of the following social conditions currently apply and might interfere with study participation, compliance, or compromise patient safety? (Select all tha… | #opts=7 | opt0: Unstable housing or homelessness


2025-12-25 18:26:46,560 [INFO] Saved questions | rows=79 | path=../output/question/ques_gpt-5-mini_AML_questions_only.xlsx
2025-12-25 18:26:46,560 [INFO] Generating option mappings...
2025-12-25 18:26:46,570 [INFO] Simplifying 407 options...
2025-12-25 18:26:53,117 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:26:58,748 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:27:07,043 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:27:13,162 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:27:19,349 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:27:26,605 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-25 18:27:31,866 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions